# День 1 — Качество данных, схема связей и аналитическая база уровня заказа

## Контекст проекта

Этот ноутбук — первый этап проекта **Marketplace Product & Financial Analytics: Olist E-commerce Case**.

Цель Дня 1 — разобраться в структуре датасетов Olist, проверить качество данных, изучить связи между таблицами и подготовить корректную аналитическую базу уровня заказа для дальнейшего анализа продуктовых, финансовых, retention-, seller funnel- и ML-метрик.

Главный риск в этих данных — разные уровни детализации таблиц. Например, таблица `orders` содержит одну строку на заказ, а таблицы `order_items` и `payments` могут содержать несколько строк на один заказ. Поэтому прямое соединение таблиц без предварительной агрегации может привести к задвоению выручки и искажению бизнес-метрик.

К концу этого ноутбука будет подготовлена основная аналитическая витрина:

`orders_base`

Эта таблица станет фундаментом для следующих этапов проекта.

## 1. Настройка окружения

В этом разделе подключим необходимые библиотеки, зададим настройки отображения и определим пути к папкам проекта.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [3]:
PROJECT_ROOT = Path("..")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

RAW_DATA_PATH

WindowsPath('../data/raw')

## 2. Загрузка данных

Сначала проверим, какие CSV-файлы находятся в папке `data/raw`, чтобы убедиться, что все необходимые таблицы доступны для анализа.

In [4]:
csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))

for file in csv_files:
    print(file.name)

olist_closed_deals_dataset.csv
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_marketing_qualified_leads_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


### 2.1. Загрузка таблиц

Загрузим все CSV-файлы в отдельные датафреймы. Для удобства будем использовать короткие и понятные названия таблиц: `orders`, `order_items`, `payments`, `reviews`, `customers`, `products`, `sellers`, `geolocation`, `category_translation`, `mql` и `closed_deals`.

In [5]:
orders = pd.read_csv(RAW_DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DATA_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DATA_PATH / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW_DATA_PATH / "olist_customers_dataset.csv")
products = pd.read_csv(RAW_DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DATA_PATH / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_DATA_PATH / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_DATA_PATH / "product_category_name_translation.csv")

mql = pd.read_csv(RAW_DATA_PATH / "olist_marketing_qualified_leads_dataset.csv")
closed_deals = pd.read_csv(RAW_DATA_PATH / "olist_closed_deals_dataset.csv")

In [6]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "customers": customers,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
    "mql": mql,
    "closed_deals": closed_deals,
}

table_shapes = pd.DataFrame(
    [
        {
            "table_name": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
        }
        for name, df in tables.items()
    ]
).sort_values("rows", ascending=False)

table_shapes

,table_name,rows,columns
7,geolocation,1000163,5
1,order_items,112650,7
2,payments,103886,5
0,orders,99441,8
4,customers,99441,5
3,reviews,99224,7
5,products,32951,9
9,mql,8000,4
6,sellers,3095,4
10,closed_deals,842,14


### Вывод по загрузке данных

Все 11 таблиц успешно загружены.

Основной e-commerce датасет содержит таблицы заказов, клиентов, товаров, продавцов, платежей, отзывов и географии. Дополнительно загружены таблицы seller marketing funnel: маркетинговые лиды продавцов и закрытые сделки.

Уже на этапе проверки размеров таблиц видно, что данные имеют разный уровень детализации:

* `orders` содержит 99 441 строку — это таблица уровня заказа;
* `customers` также содержит 99 441 строку, что соответствует связи одного заказа с одним `customer_id`;
* `order_items` содержит 112 650 строк, то есть один заказ может включать несколько товарных позиций;
* `payments` содержит 103 886 строк, то есть один заказ может иметь несколько платежных записей;
* `reviews` содержит 99 224 строки, то есть отзывы есть не для всех заказов или возможны особенности структуры отзывов;
* `geolocation` содержит более 1 млн строк и требует отдельной осторожности при использовании, так как это справочная географическая таблица.

Следовательно, перед расчётом финансовых и продуктовых метрик необходимо явно учитывать уровень детализации каждой таблицы и агрегировать таблицы `order_items` и `payments` до уровня `order_id`.


In [7]:
table_overview = []

for name, df in tables.items():
    for column in df.columns:
        table_overview.append({
            "table_name": name,
            "column_name": column,
            "dtype": df[column].dtype,
            "non_null_count": df[column].notna().sum(),
            "null_count": df[column].isna().sum(),
            "null_share": df[column].isna().mean()
        })

table_overview = pd.DataFrame(table_overview)

table_overview

,table_name,column_name,dtype,non_null_count,null_count,null_share
0,orders,order_id,str,99441,0,0.000000
1,orders,customer_id,str,99441,0,0.000000
2,orders,order_status,str,99441,0,0.000000
3,orders,order_purchase_timestamp,str,99441,0,0.000000
4,orders,order_approved_at,str,99281,160,0.001609
...,...,...,...,...,...,...
65,closed_deals,has_gtin,object,64,778,0.923990
66,closed_deals,average_stock,str,66,776,0.921615
67,closed_deals,business_type,str,832,10,0.011876
68,closed_deals,declared_product_catalog_size,float64,69,773,0.918052


### 2.2. Приведение временных колонок к формату даты

После загрузки CSV-файлов временные колонки были считаны как строки. Для дальнейшего анализа их нужно привести к типу `datetime`: это позволит рассчитывать месячную динамику, сроки доставки, задержки и cohort retention.

In [8]:
orders_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

reviews_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

order_items_date_columns = [
    "shipping_limit_date"
]

mql_date_columns = [
    "first_contact_date"
]

closed_deals_date_columns = [
    "won_date"
]

for column in orders_date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

for column in reviews_date_columns:
    reviews[column] = pd.to_datetime(reviews[column], errors="coerce")

for column in order_items_date_columns:
    order_items[column] = pd.to_datetime(order_items[column], errors="coerce")

for column in mql_date_columns:
    mql[column] = pd.to_datetime(mql[column], errors="coerce")

for column in closed_deals_date_columns:
    closed_deals[column] = pd.to_datetime(closed_deals[column], errors="coerce")

In [9]:
date_type_check = pd.DataFrame({
    "orders": orders[orders_date_columns].dtypes,
    "reviews": reviews[reviews_date_columns].dtypes,
    "order_items": order_items[order_items_date_columns].dtypes,
    "mql": mql[mql_date_columns].dtypes,
    "closed_deals": closed_deals[closed_deals_date_columns].dtypes,
})

date_type_check

,orders,reviews,order_items,mql,closed_deals
first_contact_date,NaN,NaN,NaN,datetime64[us],NaN
order_approved_at,datetime64[us],NaN,NaN,NaN,NaN
order_delivered_carrier_date,datetime64[us],NaN,NaN,NaN,NaN
order_delivered_customer_date,datetime64[us],NaN,NaN,NaN,NaN
order_estimated_delivery_date,datetime64[us],NaN,NaN,NaN,NaN
order_purchase_timestamp,datetime64[us],NaN,NaN,NaN,NaN
review_answer_timestamp,NaN,datetime64[us],NaN,NaN,NaN
review_creation_date,NaN,datetime64[us],NaN,NaN,NaN
shipping_limit_date,NaN,NaN,datetime64[us],NaN,NaN
won_date,NaN,NaN,NaN,NaN,datetime64[us]


### Вывод по типам дат

Временные колонки были успешно приведены к формату `datetime`.

Это важно для следующих этапов проекта, потому что даты будут использоваться для расчёта:

* месячной динамики заказов и GMV;
* сроков доставки;
* задержек относительно обещанной даты доставки;
* когорт клиентов;
* seller funnel по датам первого контакта и закрытия сделки.

На этом этапе данные готовы к проверке ключей, связей между таблицами и уровней детализации.


In [10]:
granularity_check = pd.DataFrame([
    {
        "table_name": "orders",
        "rows": len(orders),
        "main_key": "order_id",
        "unique_main_key": orders["order_id"].nunique(),
    },
    {
        "table_name": "customers",
        "rows": len(customers),
        "main_key": "customer_id",
        "unique_main_key": customers["customer_id"].nunique(),
    },
    {
        "table_name": "order_items",
        "rows": len(order_items),
        "main_key": "order_id",
        "unique_main_key": order_items["order_id"].nunique(),
    },
    {
        "table_name": "payments",
        "rows": len(payments),
        "main_key": "order_id",
        "unique_main_key": payments["order_id"].nunique(),
    },
    {
        "table_name": "reviews",
        "rows": len(reviews),
        "main_key": "order_id",
        "unique_main_key": reviews["order_id"].nunique(),
    },
    {
        "table_name": "products",
        "rows": len(products),
        "main_key": "product_id",
        "unique_main_key": products["product_id"].nunique(),
    },
    {
        "table_name": "sellers",
        "rows": len(sellers),
        "main_key": "seller_id",
        "unique_main_key": sellers["seller_id"].nunique(),
    },
    {
        "table_name": "mql",
        "rows": len(mql),
        "main_key": "mql_id",
        "unique_main_key": mql["mql_id"].nunique(),
    },
    {
        "table_name": "closed_deals",
        "rows": len(closed_deals),
        "main_key": "mql_id",
        "unique_main_key": closed_deals["mql_id"].nunique(),
    },
])

granularity_check["rows_minus_unique_key"] = (
    granularity_check["rows"] - granularity_check["unique_main_key"]
)

granularity_check

,table_name,rows,main_key,unique_main_key,rows_minus_unique_key
0,orders,99441,order_id,99441,0
1,customers,99441,customer_id,99441,0
2,order_items,112650,order_id,98666,13984
3,payments,103886,order_id,99440,4446
4,reviews,99224,order_id,98673,551
5,products,32951,product_id,32951,0
6,sellers,3095,seller_id,3095,0
7,mql,8000,mql_id,8000,0
8,closed_deals,842,mql_id,842,0


### Вывод по уровню детализации таблиц

Проверка ключей подтвердила, что таблицы имеют разные уровни детализации.

Таблица `orders` находится на уровне одного заказа: каждая строка соответствует одному уникальному `order_id`.

Таблица `customers` находится на уровне `customer_id`. Важно отличать `customer_id` от `customer_unique_id`: первый связан с конкретным заказом, а второй позволяет анализировать повторные покупки одного и того же клиента.

Таблица `order_items` не уникальна по `order_id`: один заказ может содержать несколько товарных позиций. Поэтому перед соединением с `orders` её нужно агрегировать до уровня заказа.

Таблица `payments` также не уникальна по `order_id`: один заказ может иметь несколько платежных записей. Перед соединением с `orders` её тоже нужно агрегировать до уровня заказа.

Таблица `reviews` почти находится на уровне заказа, но есть заказы с несколькими review-записями. Поэтому перед добавлением `review_score` в order-level витрину нужно отдельно решить, как агрегировать отзывы.

Главный методологический риск: если напрямую соединить `orders`, `order_items` и `payments`, строки заказов размножатся, а финансовые метрики будут искажены.


In [11]:
order_level_counts = pd.DataFrame({
    "items_per_order": order_items.groupby("order_id").size(),
    "payments_per_order": payments.groupby("order_id").size(),
    "reviews_per_order": reviews.groupby("order_id").size(),
})

order_level_counts.describe()

,items_per_order,payments_per_order,reviews_per_order
count,98666.000000,99440.000000,98673.000000
mean,1.141731,1.044710,1.005584
std,0.538452,0.381166,0.075060
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000
max,21.000000,29.000000,3.000000


### Вывод по количеству записей на один заказ

Большинство заказов имеют одну товарную позицию, одну платежную запись и один отзыв. Однако в данных есть заказы с несколькими строками в таблицах `order_items`, `payments` и `reviews`.

Максимальное количество товарных строк на один заказ — 21, максимальное количество платежных записей — 29, максимальное количество review-записей — 3.

Это подтверждает, что таблицы `order_items`, `payments` и `reviews` необходимо агрегировать до уровня `order_id` перед соединением с таблицей `orders`.

Особенно важно учитывать это при расчёте GMV, product revenue, freight revenue и payment value: прямой join между `orders`, `order_items` и `payments` может привести к размножению строк и завышению финансовых метрик.


In [12]:
high_multiplicity_orders = pd.DataFrame({
    "items_count": order_items.groupby("order_id").size(),
    "payments_count": payments.groupby("order_id").size(),
    "reviews_count": reviews.groupby("order_id").size(),
}).fillna(0).astype(int)

high_multiplicity_orders.sort_values(
    ["items_count", "payments_count", "reviews_count"],
    ascending=False
).head(10)

,items_count,payments_count,reviews_count
order_id,,,
8272b63d03f5f79c56e9e4120aec44ef,21,1,1
1b15974a0141d54e36626dca3fdc731a,20,1,1
ab14fdcfbe524636d65ee38360e22ce8,20,1,0
428a2f660dc84138d969ccd69a0ab6d5,15,1,1
9ef13efd6949e4573a18964dd1bbe7f5,15,1,1
73c8ab38f07dc94389065f7eba4f297a,14,1,1
9bdc4d4c71aa1de4606060929dee888c,14,1,1
37ee401157a3a0b28c9c6d0ed8c3b24b,13,1,1
2c2a19b5703863c908512d135aa6accc,12,1,1


### Вывод по заказам с большим количеством товарных строк

Проверка заказов с наибольшим количеством строк показала, что в данных действительно есть multi-item orders: один заказ может содержать 10, 15, 20 и более товарных строк.

Это нормальная особенность e-commerce данных, а не ошибка сама по себе. Однако она подтверждает необходимость предварительной агрегации `order_items` до уровня `order_id`.

Для order-level витрины такие заказы должны быть представлены одной строкой, а товарные показатели должны быть агрегированы: количество товарных строк, количество уникальных товаров, количество продавцов, сумма стоимости товаров и сумма доставки.


In [13]:
relationship_check = pd.DataFrame([
    {
        "check": "orders without items",
        "count": (~orders["order_id"].isin(order_items["order_id"])).sum()
    },
    {
        "check": "orders without payments",
        "count": (~orders["order_id"].isin(payments["order_id"])).sum()
    },
    {
        "check": "orders without reviews",
        "count": (~orders["order_id"].isin(reviews["order_id"])).sum()
    },
    {
        "check": "items without matching order",
        "count": (~order_items["order_id"].isin(orders["order_id"])).sum()
    },
    {
        "check": "payments without matching order",
        "count": (~payments["order_id"].isin(orders["order_id"])).sum()
    },
    {
        "check": "reviews without matching order",
        "count": (~reviews["order_id"].isin(orders["order_id"])).sum()
    },
])

relationship_check

,check,count
0,orders without items,775
1,orders without payments,1
2,orders without reviews,768
3,items without matching order,0
4,payments without matching order,0
5,reviews without matching order,0


### Вывод по связям между таблицами

Проверка связей показала, что в таблицах `order_items`, `payments` и `reviews` нет записей, которые ссылаются на несуществующие заказы. Это хороший признак целостности данных.

При этом часть заказов из таблицы `orders` не имеет связанных записей:

* 775 заказов не имеют товарных строк в `order_items`;
* 1 заказ не имеет платежной записи в `payments`;
* 768 заказов не имеют отзывов в `reviews`.

Такие случаи не следует автоматически считать ошибками. Для e-commerce данных это может быть связано со статусами заказов: отменённые, недоступные или незавершённые заказы могут не иметь товаров, оплат или отзывов.

Перед сборкой order-level витрины нужно проверить, с какими статусами связаны эти пропущенные связи.


In [14]:
missing_items_order_ids = orders.loc[
    ~orders["order_id"].isin(order_items["order_id"]),
    "order_id"
]

missing_payments_order_ids = orders.loc[
    ~orders["order_id"].isin(payments["order_id"]),
    "order_id"
]

missing_reviews_order_ids = orders.loc[
    ~orders["order_id"].isin(reviews["order_id"]),
    "order_id"
]

missing_relationships_by_status = pd.concat(
    [
        orders.loc[orders["order_id"].isin(missing_items_order_ids), "order_status"]
        .value_counts()
        .rename("orders_without_items"),

        orders.loc[orders["order_id"].isin(missing_payments_order_ids), "order_status"]
        .value_counts()
        .rename("orders_without_payments"),

        orders.loc[orders["order_id"].isin(missing_reviews_order_ids), "order_status"]
        .value_counts()
        .rename("orders_without_reviews"),
    ],
    axis=1
).fillna(0).astype(int)

missing_relationships_by_status

,orders_without_items,orders_without_payments,orders_without_reviews
order_status,,,
unavailable,603,0,14
canceled,164,0,20
created,5,0,2
invoiced,2,0,5
shipped,1,0,75
delivered,0,1,646
processing,0,0,6


### Вывод по пропущенным связям и статусам заказов

Проверка показала, что большинство заказов без товарных строк относятся к статусам `unavailable` и `canceled`. Это логично: если заказ был отменён или товар оказался недоступен, он может отсутствовать в таблице `order_items`.

Заказы без отзывов встречаются в разных статусах, включая `delivered`. Это не является ошибкой: клиент мог получить заказ, но не оставить отзыв.

Также обнаружен один доставленный заказ без платежной записи. Это редкий edge case, который нужно учитывать при сборке витрины, но его масштаб незначителен относительно общего объёма данных.

Для дальнейших финансовых и продуктовых метрик важно явно фильтровать заказы по статусу. Например, для расчёта GMV и клиентского опыта основной фокус должен быть на заказах со статусом `delivered`, а отменённые и недоступные заказы лучше анализировать отдельно.


In [15]:
order_status_distribution = (
    orders["order_status"]
    .value_counts()
    .rename_axis("order_status")
    .reset_index(name="orders_count")
)

order_status_distribution["orders_share"] = (
    order_status_distribution["orders_count"] / len(orders)
)

order_status_distribution

,order_status,orders_count,orders_share
0,delivered,96478,0.970203
1,shipped,1107,0.011132
2,canceled,625,0.006285
3,unavailable,609,0.006124
4,invoiced,314,0.003158
5,processing,301,0.003027
6,created,5,0.000050
7,approved,2,0.000020


### Вывод по статусам заказов

Большинство заказов в датасете имеют статус `delivered`: 96 478 заказов, или около 97% всех записей.

Это означает, что основной продуктовый, финансовый и клиентский анализ можно строить вокруг доставленных заказов, так как именно они отражают завершённый пользовательский опыт.

При этом на этапе сборки базовой order-level витрины не стоит удалять остальные статусы. Витрина `orders_base` должна сохранить колонку `order_status`, чтобы в дальнейших ноутбуках можно было явно управлять фильтрацией.

Для расчёта GMV, AOV, клиентского опыта и ML-модели плохого отзыва основной фокус будет на заказах со статусом `delivered`. Отменённые, недоступные и незавершённые заказы можно анализировать отдельно как часть операционного качества маркетплейса.


In [16]:
date_range_check = pd.DataFrame([
    {
        "table": "orders",
        "date_column": column,
        "min_date": orders[column].min(),
        "max_date": orders[column].max(),
        "null_count": orders[column].isna().sum(),
        "null_share": orders[column].isna().mean()
    }
    for column in orders_date_columns
] + [
    {
        "table": "reviews",
        "date_column": column,
        "min_date": reviews[column].min(),
        "max_date": reviews[column].max(),
        "null_count": reviews[column].isna().sum(),
        "null_share": reviews[column].isna().mean()
    }
    for column in reviews_date_columns
] + [
    {
        "table": "order_items",
        "date_column": column,
        "min_date": order_items[column].min(),
        "max_date": order_items[column].max(),
        "null_count": order_items[column].isna().sum(),
        "null_share": order_items[column].isna().mean()
    }
    for column in order_items_date_columns
] + [
    {
        "table": "mql",
        "date_column": column,
        "min_date": mql[column].min(),
        "max_date": mql[column].max(),
        "null_count": mql[column].isna().sum(),
        "null_share": mql[column].isna().mean()
    }
    for column in mql_date_columns
] + [
    {
        "table": "closed_deals",
        "date_column": column,
        "min_date": closed_deals[column].min(),
        "max_date": closed_deals[column].max(),
        "null_count": closed_deals[column].isna().sum(),
        "null_share": closed_deals[column].isna().mean()
    }
    for column in closed_deals_date_columns
])

date_range_check

,table,date_column,min_date,max_date,null_count,null_share
0,orders,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0,0.000000
1,orders,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160,0.001609
2,orders,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783,0.017930
3,orders,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965,0.029817
4,orders,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0,0.000000
5,reviews,review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00,0,0.000000
6,reviews,review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35,0,0.000000
7,order_items,shipping_limit_date,2016-09-19 00:15:34,2020-04-09 22:35:08,0,0.000000
8,mql,first_contact_date,2017-06-14 00:00:00,2018-05-31 00:00:00,0,0.000000
9,closed_deals,won_date,2017-12-05 02:00:00,2018-11-14 18:04:19,0,0.000000


### Вывод по диапазонам дат

Проверка временных колонок показала, что основная история заказов в датасете относится к периоду 2016–2018 годов.

В таблице `orders` дата покупки находится в диапазоне с сентября 2016 года по октябрь 2018 года. Это соответствует историческому характеру датасета.

В некоторых временных колонках есть пропуски:

* `order_approved_at` — заказ мог не быть подтверждён;
* `order_delivered_carrier_date` — заказ мог не быть передан перевозчику;
* `order_delivered_customer_date` — заказ мог не быть доставлен.

Такие пропуски логично связаны со статусами заказов: отменённые, недоступные или незавершённые заказы могут не иметь полной цепочки доставки.

Отдельного внимания требует колонка `shipping_limit_date` в таблице `order_items`: максимальная дата достигает 2020 года, хотя основная история заказов заканчивается в 2018 году. Этот случай нужно проверить отдельно как потенциальную временную аномалию.


In [17]:
temporal_anomalies = pd.DataFrame([
    {
        "check": "approved before purchase",
        "count": (orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum()
    },
    {
        "check": "delivered to carrier before purchase",
        "count": (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]).sum()
    },
    {
        "check": "delivered to customer before purchase",
        "count": (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum()
    },
    {
        "check": "delivered to customer before carrier",
        "count": (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()
    },
    {
        "check": "review answer before review creation",
        "count": (reviews["review_answer_timestamp"] < reviews["review_creation_date"]).sum()
    },
])

temporal_anomalies

,check,count
0,approved before purchase,0
1,delivered to carrier before purchase,166
2,delivered to customer before purchase,0
3,delivered to customer before carrier,23
4,review answer before review creation,0


In [18]:
order_items.loc[
    order_items["shipping_limit_date"] > "2018-12-31"
].sort_values("shipping_limit_date", ascending=False)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
85729,c2bb89b5c1dd978d507284be78a04cb2,1,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
85730,c2bb89b5c1dd978d507284be78a04cb2,2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,99.99,61.44
8643,13bdf405f961a6deec817d817f5c6624,1,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,69.99,14.66
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,75.99,14.70


### Вывод по временной логике

Проверка временной последовательности показала, что в большинстве случаев даты выглядят логично: нет заказов, подтверждённых до покупки, нет заказов, доставленных клиенту до покупки, и нет ответов на отзывы раньше даты создания отзыва.

При этом обнаружены отдельные временные аномалии:

* 166 заказов имеют дату передачи перевозчику раньше даты покупки;
* 23 заказа имеют дату доставки клиенту раньше даты передачи перевозчику;
* в `order_items` есть несколько строк с `shipping_limit_date` в 2020 году, хотя основная история заказов заканчивается в 2018 году.

Эти случаи выглядят как редкие ошибки или особенности записи операционных дат. Они не ломают общий анализ, но показывают, что временные признаки нужно использовать осторожно.

В дальнейших ноутбуках для анализа клиентского опыта основными временными колонками будут `order_purchase_timestamp`, `order_delivered_customer_date` и `order_estimated_delivery_date`, так как они напрямую связаны с покупкой, фактической доставкой и обещанной датой доставки.


In [19]:
numeric_quality_check = pd.DataFrame([
    {
        "table": "order_items",
        "column": "price",
        "min": order_items["price"].min(),
        "max": order_items["price"].max(),
        "mean": order_items["price"].mean(),
        "median": order_items["price"].median(),
        "zero_count": (order_items["price"] == 0).sum(),
        "negative_count": (order_items["price"] < 0).sum(),
        "null_count": order_items["price"].isna().sum()
    },
    {
        "table": "order_items",
        "column": "freight_value",
        "min": order_items["freight_value"].min(),
        "max": order_items["freight_value"].max(),
        "mean": order_items["freight_value"].mean(),
        "median": order_items["freight_value"].median(),
        "zero_count": (order_items["freight_value"] == 0).sum(),
        "negative_count": (order_items["freight_value"] < 0).sum(),
        "null_count": order_items["freight_value"].isna().sum()
    },
    {
        "table": "payments",
        "column": "payment_value",
        "min": payments["payment_value"].min(),
        "max": payments["payment_value"].max(),
        "mean": payments["payment_value"].mean(),
        "median": payments["payment_value"].median(),
        "zero_count": (payments["payment_value"] == 0).sum(),
        "negative_count": (payments["payment_value"] < 0).sum(),
        "null_count": payments["payment_value"].isna().sum()
    },
    {
        "table": "payments",
        "column": "payment_installments",
        "min": payments["payment_installments"].min(),
        "max": payments["payment_installments"].max(),
        "mean": payments["payment_installments"].mean(),
        "median": payments["payment_installments"].median(),
        "zero_count": (payments["payment_installments"] == 0).sum(),
        "negative_count": (payments["payment_installments"] < 0).sum(),
        "null_count": payments["payment_installments"].isna().sum()
    }
])

numeric_quality_check

,table,column,min,max,mean,median,zero_count,negative_count,null_count
0,order_items,price,0.85,6735.00,120.653739,74.99,0,0,0
1,order_items,freight_value,0.00,409.68,19.990320,16.26,383,0,0
2,payments,payment_value,0.00,13664.08,154.100380,100.00,9,0,0
3,payments,payment_installments,0.00,24.00,2.853349,1.00,2,0,0


### Вывод по числовым значениям

Проверка числовых колонок показала, что в ключевых финансовых полях нет отрицательных значений и пропусков.

В таблице `order_items` колонка `price` выглядит корректно: минимальная цена больше нуля, отрицательных и нулевых значений нет.

В колонке `freight_value` есть строки с нулевой стоимостью доставки. Это не обязательно ошибка: в e-commerce такие случаи могут соответствовать бесплатной доставке или особенностям промо-механик.

В таблице `payments` обнаружены редкие случаи с `payment_value = 0` и `payment_installments = 0`. Эти случаи требуют отдельной проверки, так как могут быть техническими или аномальными платежными записями.

В целом числовые данные пригодны для дальнейшего анализа, но редкие нулевые значения в платежах нужно учитывать как edge cases.


In [20]:
payments.loc[
    (payments["payment_value"] == 0) | 
    (payments["payment_installments"] == 0)
].sort_values(["payment_value", "payment_installments"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [21]:
payments.sort_values("payment_value", ascending=False).head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66
62409,2cc9089445046817a7539d90805e6e5a,1,boleto,1,6081.54
43232,a96610ab360d42a2e5335a3998b4718a,1,credit_card,10,4950.34
70320,b4c4b76c642808cbe472a32b86cddc95,1,credit_card,5,4809.44
6440,199af31afc78c699f0dbf71fb178d4d4,1,credit_card,8,4764.34
67546,8dbc85d1447242f3b127dda390d56e19,1,credit_card,8,4681.78


### Вывод по аномальным платежам

Проверка подозрительных платежных записей показала, что нулевые значения `payment_value` встречаются редко и в основном связаны с типами оплаты `voucher` и `not_defined`.

Также обнаружены две строки с `payment_installments = 0` при оплате кредитной картой. Это выглядит как техническая аномалия, так как количество платежей обычно должно быть не меньше 1.

Крупные значения `payment_value` присутствуют в данных, но сами по себе не являются ошибкой: они могут соответствовать дорогим заказам или заказам с несколькими товарами.

Для дальнейшей order-level витрины платежи будут агрегированы до уровня `order_id`. Нулевые и аномальные значения не удаляются на этапе базовой витрины, чтобы сохранить прозрачность данных, но будут учитываться при интерпретации финансовых метрик.


## 3. Сборка order-level витрины

Перед соединением таблиц необходимо привести таблицы с детализацией ниже уровня заказа к уровню `order_id`.

Начнём с таблицы `order_items`. Она содержит товарные строки заказа, поэтому один `order_id` может встречаться несколько раз. Для order-level витрины агрегируем её до одной строки на заказ.

In [22]:
order_items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        items_count=("order_item_id", "count"),
        products_count=("product_id", "nunique"),
        sellers_count=("seller_id", "nunique"),
        product_revenue=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_items_agg.head()

,order_id,items_count,products_count,sellers_count,product_revenue,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


In [23]:
order_items_agg_check = pd.DataFrame({
    "rows": [len(order_items_agg)],
    "unique_order_id": [order_items_agg["order_id"].nunique()],
    "duplicated_order_id": [order_items_agg["order_id"].duplicated().sum()]
})

order_items_agg_check

,rows,unique_order_id,duplicated_order_id
0,98666,98666,0


### Вывод по агрегации товарных строк

Таблица `order_items` была агрегирована до уровня `order_id`.

Для каждого заказа рассчитаны:

* `items_count` — количество товарных строк в заказе;
* `products_count` — количество уникальных товаров;
* `sellers_count` — количество уникальных продавцов;
* `product_revenue` — сумма стоимости товаров;
* `freight_value` — сумма стоимости доставки.

После агрегации в таблице `order_items_agg` осталось 98 666 уникальных заказов, дубликатов `order_id` нет.

Теперь эту таблицу можно безопасно соединять с `orders` без риска размножения строк из-за нескольких товарных позиций в одном заказе.


In [24]:
payments_agg = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_types_count=("payment_type", "nunique"),
        payment_records_count=("payment_sequential", "count")
    )
    .reset_index()
)

payments_agg.head()

,order_id,payment_value,payment_installments,payment_types_count,payment_records_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,1


In [25]:
payments_agg_check = pd.DataFrame({
    "rows": [len(payments_agg)],
    "unique_order_id": [payments_agg["order_id"].nunique()],
    "duplicated_order_id": [payments_agg["order_id"].duplicated().sum()]
})

payments_agg_check

,rows,unique_order_id,duplicated_order_id
0,99440,99440,0


### Вывод по агрегации платежей

Таблица `payments` была агрегирована до уровня `order_id`.

Для каждого заказа рассчитаны:

* `payment_value` — суммарная сумма платежей по заказу;
* `payment_installments` — максимальное количество платежных частей по заказу;
* `payment_types_count` — количество уникальных типов оплаты;
* `payment_records_count` — количество платежных записей.

После агрегации в таблице `payments_agg` осталось 99 440 уникальных заказов, дубликатов `order_id` нет.

Теперь платежи можно безопасно соединять с таблицей `orders` без риска размножения строк и завышения финансовых метрик.


In [26]:
reviews_agg = (
    reviews
    .sort_values("review_answer_timestamp")
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean"),
        review_records_count=("review_id", "count"),
        first_review_creation_date=("review_creation_date", "min"),
        last_review_answer_timestamp=("review_answer_timestamp", "max")
    )
    .reset_index()
)

reviews_agg.head()

,order_id,review_score,review_records_count,first_review_creation_date,last_review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1,2017-09-21,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,4.0,1,2017-05-13,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,5.0,1,2018-01-23,2018-01-23 16:06:31
3,00024acbcdf0a6daa1e931b038114c75,4.0,1,2018-08-15,2018-08-15 16:39:01
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1,2017-03-02,2017-03-03 10:54:59


In [27]:
reviews_agg_check = pd.DataFrame({
    "rows": [len(reviews_agg)],
    "unique_order_id": [reviews_agg["order_id"].nunique()],
    "duplicated_order_id": [reviews_agg["order_id"].duplicated().sum()]
})

reviews_agg_check

,rows,unique_order_id,duplicated_order_id
0,98673,98673,0


### Вывод по агрегации отзывов

Таблица `reviews` была агрегирована до уровня `order_id`.

Для каждого заказа рассчитаны:

* `review_score` — средняя оценка отзыва по заказу;
* `review_records_count` — количество review-записей;
* `first_review_creation_date` — первая дата создания отзыва;
* `last_review_answer_timestamp` — последняя дата ответа на отзыв.

После агрегации в таблице `reviews_agg` осталось 98 673 уникальных заказа, дубликатов `order_id` нет.

Такой подход позволяет безопасно добавить отзывы в order-level витрину. При этом важно помнить, что у небольшой части заказов может быть несколько review-записей, поэтому `review_records_count` сохраняется как диагностическая колонка.


In [28]:
orders_base = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(order_items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

orders_base.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,items_count,products_count,sellers_count,product_revenue,freight_value,payment_value,payment_installments,payment_types_count,payment_records_count,review_score,review_records_count,first_review_creation_date,last_review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,38.71,1.0,2.0,3.0,4.0,1.0,2017-10-11,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,118.70,22.76,141.46,1.0,1.0,1.0,4.0,1.0,2018-08-08,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,179.12,3.0,1.0,1.0,5.0,1.0,2018-08-18,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,72.20,1.0,1.0,1.0,5.0,1.0,2017-12-03,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,19.90,8.72,28.62,1.0,1.0,1.0,5.0,1.0,2018-02-17,2018-02-18 13:02:51


In [29]:
orders_base_check = pd.DataFrame({
    "rows": [len(orders_base)],
    "unique_order_id": [orders_base["order_id"].nunique()],
    "duplicated_order_id": [orders_base["order_id"].duplicated().sum()],
    "source_orders_rows": [len(orders)]
})

orders_base_check

,rows,unique_order_id,duplicated_order_id,source_orders_rows
0,99441,99441,0,99441


### Вывод по сборке базовой order-level витрины

Базовая витрина `orders_base` была собрана на уровне одного заказа.

Перед соединением таблиц данные из `order_items`, `payments` и `reviews` были предварительно агрегированы до уровня `order_id`. Это позволило избежать размножения строк и искажения финансовых метрик.

Проверка показала, что итоговая витрина содержит 99 441 строку, 99 441 уникальный `order_id` и не содержит дубликатов заказов.

Таким образом, главный методологический риск Дня 1 — задвоение выручки из-за join таблиц разной детализации — был устранён.


In [30]:
orders_base["gmv"] = orders_base["product_revenue"]

orders_base["order_value_with_freight"] = (
    orders_base["product_revenue"] + orders_base["freight_value"]
)

orders_base["delivery_days"] = (
    orders_base["order_delivered_customer_date"] 
    - orders_base["order_purchase_timestamp"]
).dt.days

orders_base["delivery_delay_days"] = (
    orders_base["order_delivered_customer_date"] 
    - orders_base["order_estimated_delivery_date"]
).dt.days

orders_base["is_delivered"] = orders_base["order_status"].eq("delivered")
orders_base["has_review"] = orders_base["review_score"].notna()
orders_base["has_payment"] = orders_base["payment_value"].notna()
orders_base["has_items"] = orders_base["product_revenue"].notna()

orders_base.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,items_count,products_count,sellers_count,product_revenue,freight_value,payment_value,payment_installments,payment_types_count,payment_records_count,review_score,review_records_count,first_review_creation_date,last_review_answer_timestamp,gmv,order_value_with_freight,delivery_days,delivery_delay_days,is_delivered,has_review,has_payment,has_items
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,38.71,1.0,2.0,3.0,4.0,1.0,2017-10-11,2017-10-12 03:43:48,29.99,38.71,8.0,-8.0,True,True,True,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,118.70,22.76,141.46,1.0,1.0,1.0,4.0,1.0,2018-08-08,2018-08-08 18:37:50,118.70,141.46,13.0,-6.0,True,True,True,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,179.12,3.0,1.0,1.0,5.0,1.0,2018-08-18,2018-08-22 19:07:58,159.90,179.12,9.0,-18.0,True,True,True,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,72.20,1.0,1.0,1.0,5.0,1.0,2017-12-03,2017-12-05 19:21:58,45.00,72.20,13.0,-13.0,True,True,True,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,19.90,8.72,28.62,1.0,1.0,1.0,5.0,1.0,2018-02-17,2018-02-18 13:02:51,19.90,28.62,2.0,-10.0,True,True,True,True


In [31]:
orders_base.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'items_count',
 'products_count',
 'sellers_count',
 'product_revenue',
 'freight_value',
 'payment_value',
 'payment_installments',
 'payment_types_count',
 'payment_records_count',
 'review_score',
 'review_records_count',
 'first_review_creation_date',
 'last_review_answer_timestamp',
 'gmv',
 'order_value_with_freight',
 'delivery_days',
 'delivery_delay_days',
 'is_delivered',
 'has_review',
 'has_payment',
 'has_items']

### Вывод по структуре `orders_base`

Витрина `orders_base` содержит одну строку на заказ и объединяет ключевую информацию из нескольких таблиц:

* данные заказа и статуса доставки;
* идентификаторы клиента;
* географию клиента;
* агрегированные показатели по товарам;
* агрегированные показатели по платежам;
* агрегированные показатели по отзывам;
* расчётные признаки доставки и наличия связанных данных.

Витрина сохраняет все заказы из исходной таблицы `orders`, включая доставленные, отменённые, недоступные и незавершённые. Это позволяет не терять данные на этапе подготовки и явно управлять фильтрацией в следующих аналитических ноутбуках.


In [32]:
orders_base_quality_check = pd.DataFrame({
    "column": orders_base.columns,
    "dtype": [orders_base[column].dtype for column in orders_base.columns],
    "null_count": [orders_base[column].isna().sum() for column in orders_base.columns],
    "null_share": [orders_base[column].isna().mean() for column in orders_base.columns],
    "unique_count": [orders_base[column].nunique(dropna=True) for column in orders_base.columns],
})

orders_base_quality_check

,column,dtype,null_count,null_share,unique_count
0,order_id,str,0,0.000000,99441
1,customer_id,str,0,0.000000,99441
2,order_status,str,0,0.000000,8
3,order_purchase_timestamp,datetime64[us],0,0.000000,98875
4,order_approved_at,datetime64[us],160,0.001609,90733
5,order_delivered_carrier_date,datetime64[us],1783,0.017930,81018
6,order_delivered_customer_date,datetime64[us],2965,0.029817,95664
7,order_estimated_delivery_date,datetime64[us],0,0.000000,459
8,customer_unique_id,str,0,0.000000,96096
9,customer_zip_code_prefix,int64,0,0.000000,14994


In [33]:
orders_base[[
    "items_count",
    "products_count",
    "sellers_count",
    "product_revenue",
    "freight_value",
    "payment_value",
    "review_score",
    "delivery_days",
    "delivery_delay_days",
    "has_items",
    "has_payment",
    "has_review",
    "is_delivered"
]].describe(include="all")

,items_count,products_count,sellers_count,product_revenue,freight_value,payment_value,review_score,delivery_days,delivery_delay_days,has_items,has_payment,has_review,is_delivered
count,98666.000000,98666.000000,98666.000000,98666.000000,98666.000000,99440.000000,98673.000000,96476.000000,96476.000000,99441,99441,99441,99441
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,2,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98666,99440,98673,96478
mean,1.141731,1.038098,1.013622,137.754076,22.823562,160.990267,4.086793,12.094086,-11.876881,NaN,NaN,NaN,NaN
std,0.538452,0.226456,0.122297,210.645145,21.650909,221.951257,1.346274,9.551746,10.183854,NaN,NaN,NaN,NaN
min,1.000000,1.000000,1.000000,0.850000,0.000000,0.000000,1.000000,0.000000,-147.000000,NaN,NaN,NaN,NaN
25%,1.000000,1.000000,1.000000,45.900000,13.850000,62.010000,4.000000,6.000000,-17.000000,NaN,NaN,NaN,NaN
50%,1.000000,1.000000,1.000000,86.900000,17.170000,105.290000,5.000000,10.000000,-12.000000,NaN,NaN,NaN,NaN
75%,1.000000,1.000000,1.000000,149.900000,24.040000,176.970000,5.000000,15.000000,-7.000000,NaN,NaN,NaN,NaN


### Вывод по качеству итоговой витрины

Проверка `orders_base` показала, что витрина собрана корректно: количество строк совпадает с исходной таблицей `orders`, каждый `order_id` уникален, дубликатов заказов нет.

Пропуски в товарных, платежных и review-полях объясняются отсутствием связанных записей в исходных таблицах. Это ожидаемо для заказов со статусами `canceled`, `unavailable`, а также для заказов без отзывов.

Средняя задержка доставки отрицательная, что означает, что значительная часть заказов была доставлена раньше обещанной даты. При этом есть и положительные значения `delivery_delay_days`, которые в следующих ноутбуках можно использовать для анализа плохого клиентского опыта.

Витрина подходит для дальнейшего анализа, но перед сохранением нужно явно обработать пропуски в агрегированных числовых колонках: если у заказа нет товарных или платежных строк, соответствующие счётчики и суммы можно заполнить нулями, сохранив флаги `has_items`, `has_payment` и `has_review`.


In [34]:
fill_zero_columns = [
    "items_count",
    "products_count",
    "sellers_count",
    "product_revenue",
    "freight_value",
    "payment_value",
    "payment_installments",
    "payment_types_count",
    "payment_records_count",
    "review_records_count",
    "gmv",
    "order_value_with_freight",
]

orders_base[fill_zero_columns] = orders_base[fill_zero_columns].fillna(0)

In [35]:
orders_base[fill_zero_columns].isna().sum()

items_count                 0
products_count              0
sellers_count               0
product_revenue             0
freight_value               0
payment_value               0
payment_installments        0
payment_types_count         0
payment_records_count       0
review_records_count        0
gmv                         0
order_value_with_freight    0
dtype: int64

### Вывод по обработке пропусков в агрегированных полях

Пропуски в агрегированных числовых колонках были заполнены нулями.

Это сделано только для тех полей, где отсутствие связанной записи означает отсутствие соответствующей сущности или суммы: товарных строк, платежных записей, review-записей, товарной выручки и стоимости доставки.

При этом логические флаги `has_items`, `has_payment` и `has_review` были созданы до заполнения пропусков, поэтому информация о наличии или отсутствии связанных данных сохранена.


In [36]:
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

orders_base.to_csv(
    PROCESSED_DATA_PATH / "orders_base.csv",
    index=False
)

In [37]:
sorted(PROCESSED_DATA_PATH.glob("*.csv"))

[WindowsPath('../data/processed/orders_base.csv')]

## Итог Дня 1

В рамках первого дня проекта была проведена базовая проверка качества данных, изучена структура таблиц Olist и собрана первая аналитическая витрина уровня заказа — `orders_base`.

Основные результаты:

1. Были загружены все 11 таблиц проекта: основной e-commerce dataset и seller marketing funnel dataset.
2. Проверены размеры таблиц, типы данных, пропуски, ключи и уровни детализации.
3. Временные колонки были приведены к формату `datetime`.
4. Подтверждено, что таблицы имеют разный уровень детализации:

   * `orders` — одна строка на заказ;
   * `order_items` — несколько товарных строк на заказ;
   * `payments` — несколько платежных записей на заказ;
   * `reviews` — в редких случаях несколько отзывов на заказ.
5. Проверены связи между таблицами: не найдено записей в `order_items`, `payments` или `reviews`, которые ссылаются на несуществующие заказы.
6. Проверены статусы заказов и пропущенные связи. Большинство заказов без товарных строк относятся к статусам `unavailable` и `canceled`.
7. Проверены временные и числовые аномалии. Критичных проблем, мешающих дальнейшему анализу, не обнаружено, но отдельные edge cases зафиксированы как ограничения данных.
8. Таблицы `order_items`, `payments` и `reviews` были агрегированы до уровня `order_id`.
9. Собрана витрина `orders_base`, содержащая 99 441 строку и 99 441 уникальный заказ.
10. Проверено, что после join не возникло дубликатов `order_id`, а значит удалось избежать главного риска — задвоения выручки.
11. Витрина сохранена в файл `data/processed/orders_base.csv`.

Главный методологический вывод:

Данные Olist имеют нормализованную структуру, поэтому перед расчётом продуктовых и финансовых метрик необходимо явно контролировать уровень детализации таблиц. Прямое соединение `orders`, `order_items` и `payments` без предварительной агрегации могло бы привести к размножению строк и завышению выручки.

Витрина `orders_base` станет основой для следующих этапов проекта: расчёта GMV, AOV, динамики заказов, анализа категорий, retention, RFM, seller funnel и ML-модели риска плохого клиентского опыта.
